# Módulo 5 · Filtrado, *notch* y remuestreo

**Objetivo:** limpiar la señal de lo que no es cerebro (deriva lenta, red eléctrica,
ruido de alta frecuencia) y **unificar la frecuencia de muestreo**, imprescindible en TUSZ,
donde conviven 250, 256, 400, 512 y 1000 Hz.

Orden recomendado del preprocesado (lo que hace `tusz_utils.preprocess`):

```
EDF → limpiar canales → TCP → notch 60 Hz → paso banda (0.5-40 Hz) → remuestreo (200 Hz)
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from tusz_utils import read_tusz_raw, make_tcp, INVENTORY

mne.set_log_level("WARNING")
mne.viz.set_browser_backend("matplotlib")

raw = make_tcp(read_tusz_raw("aaaaabqv_s001_t000"))
raw

## 5.1 El punto de partida

In [ ]:
def psd_plot(r, title, ax=None, fmax=None):
    fmax = fmax or r.info["sfreq"] / 2
    p, f = r.compute_psd(fmax=fmax).get_data(return_freqs=True)
    ax = ax or plt.gca()
    ax.semilogy(f, p.mean(0) * 1e12, lw=1)
    ax.set_title(title); ax.set_xlabel("Hz"); ax.set_ylabel("µV²/Hz")

plt.figure(figsize=(8, 3.5)); psd_plot(raw, "Sin filtrar (TCP, media de canales)"); plt.show()

Problemas visibles: potencia enorme por debajo de 0.5 Hz (deriva, sudor, movimiento) y el pico
de **60 Hz** (+ su armónico 120 Hz cerca de Nyquist).

## 5.2 Filtro *notch* para la red eléctrica

In [ ]:
raw_n = raw.copy().notch_filter(freqs=[60, 120] if raw.info["sfreq"] > 240 else [60])
fig, axs = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
psd_plot(raw, "Antes", axs[0]); psd_plot(raw_n, "Después del notch 60 Hz", axs[1]); plt.show()

> Si vas a aplicar después un paso bajo a 40 Hz, el notch de 60 Hz es casi redundante; aun así se
> suele aplicar (no cuesta nada y reduce el *ringing* del filtro paso bajo con una línea tan potente).

## 5.3 Filtro paso banda: `raw.filter(l_freq, h_freq)`

- `l_freq` = paso alto (quita lo más lento). Típico en crisis: **0.5 Hz** (o 1 Hz).
- `h_freq` = paso bajo (quita lo más rápido). Típico: **40-50 Hz**; hay trabajos que usan hasta 70-100 Hz
  para captar actividad rápida al inicio de crisis focales.
- Por defecto MNE usa un **FIR de fase cero** (no desplaza la señal en el tiempo): ideal.

In [ ]:
raw_f = raw_n.copy().filter(l_freq=0.5, h_freq=40)
fig, axs = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
psd_plot(raw_n, "Solo notch", axs[0]); psd_plot(raw_f, "Notch + paso banda 0.5-40 Hz", axs[1]); plt.show()

### ¿Cómo es el filtro por dentro?
`mne.filter.create_filter` + `mne.viz.plot_filter` muestran la respuesta en frecuencia y en tiempo.
Muy útil para justificar elecciones en la memoria.

In [ ]:
h = mne.filter.create_filter(raw.get_data()[:1], raw.info["sfreq"], l_freq=0.5, h_freq=40, verbose=False)
print("Longitud del filtro FIR:", len(h), "muestras =", len(h) / raw.info["sfreq"], "s")
fig = mne.viz.plot_filter(h, raw.info["sfreq"], freq=None, gain=None, fscale="linear",
                          flim=(0, 60), title="FIR 0.5-40 Hz")

> El filtro dura ~6.6 s: un paso alto a 0.5 Hz necesita mucho "contexto". Por eso **se filtra
> la señal continua antes de cortar ventanas**, nunca cada ventana de 4 s por separado.

### Efecto en la señal

In [ ]:
t0, t1 = 170, 195
ch = "FP1-F7"
a, t = raw.get_data(picks=ch, tmin=t0, tmax=t1, return_times=True, units="uV")
b = raw_f.get_data(picks=ch, tmin=t0, tmax=t1, units="uV")
plt.figure(figsize=(12, 3.5))
plt.plot(t, a[0], lw=0.6, label="original", alpha=0.7)
plt.plot(t, b[0], lw=0.8, label="filtrado 0.5-40 Hz + notch")
plt.axvspan(179.884, 187.016, color="tab:red", alpha=0.1)
plt.legend(); plt.xlabel("s"); plt.ylabel("µV"); plt.title(ch); plt.show()

### IIR (Butterworth) como alternativa
Más corto y barato, pero con distorsión de fase si no se aplica hacia delante y hacia atrás.
MNE ya lo hace de fase cero (`filtfilt`) por defecto con `method="iir"`:

In [ ]:
raw_iir = raw.copy().filter(0.5, 40, method="iir", iir_params=dict(order=4, ftype="butter"))
print(raw_iir.info["highpass"], raw_iir.info["lowpass"])

## 5.4 Remuestreo: unificar `sfreq`

Una red neuronal espera siempre el **mismo número de muestras por ventana**. En TUSZ:

In [ ]:
files = pd.read_csv(INVENTORY / "files.csv")
files.fs.value_counts().rename("nº ficheros").to_frame()

Solución: remuestrear todo a una frecuencia común. **200 Hz** es una elección frecuente
(Nyquist 100 Hz > tu paso bajo de 40 Hz con mucho margen) y reduce la memoria un 20 % respecto
a 250 Hz. Otras opciones típicas: 128 Hz o 256 Hz.

In [ ]:
raw_200 = raw_f.copy().resample(200)
print(raw_f.info["sfreq"], "Hz →", raw_200.info["sfreq"], "Hz |", raw_f.n_times, "→", raw_200.n_times, "muestras")
print("Anotaciones conservadas:", raw_200.annotations)

> `resample` aplica su propio filtro anti-*aliasing*, pero es buena práctica filtrar paso bajo **antes**.
> Las anotaciones están en segundos, así que no se ven afectadas.

## 5.5 La función de preprocesado del curso

In [ ]:
from tusz_utils import preprocess
import inspect
print(inspect.getsource(preprocess))

In [ ]:
clean = preprocess(read_tusz_raw("aaaaabqv_s001_t000"))
clean

Guarda el resultado en FIF: preprocesar los 81 GB de TUSZ es lento, hazlo **una vez** y luego
trabaja siempre con los `.fif` limpios.

In [ ]:
from tusz_utils import SALIDAS
clean.save(SALIDAS / "aaaaabqv_s001_t000_prep_raw.fif", overwrite=True)

---
## 🧪 Ejercicios

**E5.1** Compara el PSD de la 1ª crisis con paso bajo a 40 Hz y a 70 Hz. ¿Se pierde algo relevante
en una ausencia?

**E5.2** Preprocesa `aaaaamtc_s001_t000` (250 Hz, AR) y comprueba que el resultado tiene exactamente
la misma forma (`n_canales`, `sfreq`) que `clean`.

**E5.3** ¿Cuánto tarda `preprocess` en un fichero de ~24 min? Mídelo con `time.perf_counter()` y
estima cuánto tardarías en preprocesar todo TUSZ (~1475 h). (Pista: paraleliza con `joblib`.)

### Solución E5.1

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for hf in (40, 70):
    r = make_tcp(read_tusz_raw("aaaaabqv_s001_t000")).filter(0.5, hf).crop(180, 187)
    p, f = r.compute_psd(fmax=90, n_fft=512).get_data(return_freqs=True)
    ax.semilogy(f, p.mean(0) * 1e12, label=f"0.5-{hf} Hz")
ax.legend(); ax.set_xlabel("Hz"); plt.show()
# En la ausencia casi toda la energía está < 20 Hz: 40 Hz no pierde nada importante.

### Solución E5.2

In [ ]:
c2 = preprocess(read_tusz_raw("aaaaamtc_s001_t000"))
print(len(c2.ch_names), c2.info["sfreq"], "|", len(clean.ch_names), clean.info["sfreq"])

### Solución E5.3

In [ ]:
import time
t = time.perf_counter()
_ = preprocess(read_tusz_raw("aaaaabqv_s001_t000"))
dt = time.perf_counter() - t
horas_fichero = 1437 / 3600
print(f"{dt:.1f} s por fichero → ~{dt / horas_fichero * 1475 / 3600:.1f} h para todo TUSZ en un núcleo")